# 测试

In [1]:
import env

In [2]:
import numpy as np

import tvm
from tvm import te
from tvm import rpc
from tvm.contrib import utils

n = tvm.runtime.convert(1024)
A = te.placeholder((n,), name="A")
B = te.compute((n,), lambda i: A[i] + 1.0, name="B")
s = te.create_schedule(B.op)

In [3]:
!gcc -v

Using built-in specs.
COLLECT_GCC=gcc
COLLECT_LTO_WRAPPER=/usr/lib/gcc/x86_64-linux-gnu/11/lto-wrapper
OFFLOAD_TARGET_NAMES=nvptx-none:amdgcn-amdhsa
OFFLOAD_TARGET_DEFAULT=1
Target: x86_64-linux-gnu
Configured with: ../src/configure -v --with-pkgversion='Ubuntu 11.4.0-2ubuntu1~20.04' --with-bugurl=file:///usr/share/doc/gcc-11/README.Bugs --enable-languages=c,ada,c++,go,brig,d,fortran,objc,obj-c++,m2 --prefix=/usr --with-gcc-major-version-only --program-suffix=-11 --program-prefix=x86_64-linux-gnu- --enable-shared --enable-linker-build-id --libexecdir=/usr/lib --without-included-gettext --enable-threads=posix --libdir=/usr/lib --enable-nls --enable-bootstrap --enable-clocale=gnu --enable-libstdcxx-debug --enable-libstdcxx-time=yes --with-default-libstdcxx-abi=new --enable-gnu-unique-object --disable-vtable-verify --enable-plugin --enable-default-pie --with-system-zlib --enable-libphobos-checking=release --with-target-system-zlib=auto --enable-objc-gc=auto --enable-multiarch --disable-we

In [4]:
# target = "llvm"
target = "llvm -mtriple=x86_64-linux-gnu"

func = tvm.build(s, [A, B], target=target, name="add_one")
# save the lib at a local temp folder
temp = utils.tempdir()
path = temp.relpath("lib.tar")
func.export_library(path)

## 通过 RPC 远程运行CPU内核

将展示如何在远程设备上运行生成的 CPU 内核。

首先，从远程设备获取 RPC 会话。

In [5]:
# remote = rpc.LocalSession() # 仿真
# 以下是我的环境，请将其更改为您的目标设备 IP 地址。
host = "10.16.11.15"
port = 9090
remote = rpc.connect(host, port)

将库上传到远程设备，然后调用设备本地的编译器重新链接它们。现在 `func` 是远程模块对象。

In [6]:
remote.upload(path)
func = remote.load_module("lib.tar")

# 在远程设备上创建数组
dev = remote.cpu()
a = tvm.nd.array(np.random.uniform(size=1024).astype(A.dtype), dev)
b = tvm.nd.array(np.zeros(1024, dtype=A.dtype), dev)
# 该函数将在远程设备上运行。
func(a, b)
np.testing.assert_equal(b.numpy(), a.numpy() + 1)

当你想评估远程设备上内核的性能时，重要的是要避免网络开销。`time_evaluator` 将返回远程函数，该函数在远程设备上多次运行该函数，测量每次运行的成本，并返回测量到的成本。网络开销被排除在外。

In [7]:
time_f = func.time_evaluator(func.entry_name, dev, number=10)
cost = time_f(a, b).mean
print(f"{cost:g} secs/op")

1.146e-07 secs/op


## 通过 RPC 远程运行 OpenCL 内核

对于远程 OpenCL 设备，工作流程几乎与上述相同。

您可以定义内核，上传文件，并通过 RPC 运行。

```{note}
:class: alert alert-info
树莓派不支持 OpenCL，以下代码已在 Firefly-RK3399 上测试。您可以参考此 [教程](https://gist.github.com/mli/585aed2cec0b5178b1a510f9f236afa2)为RK3399设置操作系统和OpenCL驱动。
```

还需要构建在rk3399板上启用了OpenCL的运行时。在 TVM 根目录下执行以下命令：

```bash
cp cmake/config.cmake .
sed -i "s/USE_OPENCL OFF/USE_OPENCL ON/" config.cmake
make runtime -j4
```

以下函数展示了如何远程运行 OpenCL 内核：

In [8]:
def run_opencl(opencl_device_host = "10.77.1.145"):
    # NOTE: 这是我的rk3399开发板的设置。您需要根据您的环境进行相应的修改。
    opencl_device_port = 9090
    target = tvm.target.Target("opencl", host="llvm -mtriple=x86_64-linux-gnu")

    # 为上述 "add one" 计算声明创建调度
    s = te.create_schedule(B.op)
    xo, xi = s[B].split(B.op.axis[0], factor=32)
    s[B].bind(xo, te.thread_axis("blockIdx.x"))
    s[B].bind(xi, te.thread_axis("threadIdx.x"))
    func = tvm.build(s, [A, B], target=target)

    remote = rpc.connect(opencl_device_host, opencl_device_port)

    # export and upload
    path = temp.relpath("lib_cl.tar")
    func.export_library(path)
    remote.upload(path)
    func = remote.load_module("lib_cl.tar")

    # run
    dev = remote.cl()
    a = tvm.nd.array(np.random.uniform(size=1024).astype(A.dtype), dev)
    b = tvm.nd.array(np.zeros(1024, dtype=A.dtype), dev)
    func(a, b)
    np.testing.assert_equal(b.numpy(), a.numpy() + 1)
    print("OpenCL test passed!")

In [10]:
run_opencl(opencl_device_host = "10.16.11.15")

OpenCL test passed!
